In [1]:
import pandas as pd
from pathlib import Path
from neo4j import GraphDatabase
from getpass import getpass

In [2]:
URI = "bolt://127.0.0.1:7687"
USERNAME = "neo4j"
PASSWORD = getpass("Masukkan password Neo4j: ")

In [3]:
DATA_DIR = (
    Path(r"C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039")
    / "data"
    / "processed"
)

hasil_similarity = pd.read_csv(
    DATA_DIR / "top5_similarity_item_P1.csv"
)

print(hasil_similarity.to_string(index=False))

 item_acuan  item_mirip  similarity
          6      160043    0.447214
          6      236992    0.316228
          6      216965    0.316228
          6      356569    0.258199
          6      413179    0.223607


In [4]:
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

rows = hasil_similarity.to_dict(orient="records")

try:
    with driver.session(database="neo4j") as session:

        # Hapus hasil sebelumnya untuk acuan dan metode yang sama
        session.run("""
            MATCH (a:Item {itemId: $reference_id})
                  -[r:SIMILAR_TO {fold: 'P1', method: 'cosine'}]->()
            DELETE r
        """, reference_id=int(hasil_similarity.iloc[0]["item_acuan"]))

        # Tulis lima hasil similarity ke graf
        session.run("""
            UNWIND $rows AS row
            MATCH (a:Item {itemId: row.item_acuan})
            MATCH (b:Item {itemId: row.item_mirip})
            MERGE (a)-[r:SIMILAR_TO {
                fold: 'P1',
                method: 'cosine'
            }]->(b)
            SET r.similarity = row.similarity
        """, rows=rows)


        print("Proses penyimpanan selesai.")

finally:
    driver.close()

Proses penyimpanan selesai.
